# Workshop: Sometimes Less is More
## Optimising HPC Efficiency & Slurm Scaling on REANNZ Infrastructure

**Duration:** 2 Hours  
**Objective:** Learn how over-allocating CPU cores wastes budget, increases queue times, and degrades parallel efficiency. You will write, submit, and profile jobs to find the scaling 'sweet spot'.

---

## 1. Conceptual Framework & Scaling Laws (25 Mins)

Before we run jobs on the cluster, we must understand the mathematical limits of parallelism. Adding more cores does not automatically make code faster. Two core laws dictate performance limits:

### Amdahl's Law (Strong Scaling)
Amdahl's Law models a **fixed total problem size**. It calculates the theoretical maximum speedup achievable when only a fraction of the program can be parallelised.

$$
S(N) = \frac{1}{(1 - P) + \frac{P}{N}}
$$

*   **$S(N)$**: Theoretical speedup factor.
*   **$P$**: The proportion of the program that can be parallelised (value between 0 and 1).
*   **$1 - P$**: The serial fraction (e.g., I/O operations, initialization, data logging).
*   **$N$**: The number of CPU cores.

**Crucial Insight:** If 5% of your code is serial ($1-P = 0.05$), your maximum possible speedup is **20x**, even if you throw 1,000,000 cores at it. The serial bottleneck quickly dominates.

### Universal Scalability Law (USL)
Amdahl's Law assumes communication between cores is free. In reality, as you add cores on REANNZ hardware, they must talk to each other to sync data. Dr. Neil Gunther’s USL expands Amdahl's Law by accounting for **contention** (queuing for shared resources) and **crosstalk** (coherency delays).

$$
S(N) = \frac{N}{1 + \sigma(N - 1) + \kappa N(N - 1)}
$$

*   **$\sigma$ (Sigma)**: Contention penalty (fraction of time spent waiting for shared resources).
*   **$\kappa$ (Kappa)**: Crosstalk penalty (overhead that grows quadratically due to inter-core communication).

Because of $\kappa$, **your code will eventually run slower if you give it too many cores.** This point is called the retrograde scaling limit.

### Mathematical Simulation of Scaling Laws
Let's plot these mathematical frameworks across a full range of core scales (1 to 64) to see how inter-core communication destroys performance at higher core counts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cores = np.array([1, 2, 4, 8, 16, 32, 64])
P = 0.95        # 95% parallelisable code
sigma = 0.02    # 2% contention 
kappa = 0.0003  # Small crosstalk coefficient

# Calculations
amdahl_speedup = 1 / ((1 - P) + (P / cores))
usl_speedup = cores / (1 + sigma * (cores - 1) + kappa * cores * (cores - 1))

# Plotting the comparison
plt.figure(figsize=(10, 6))
plt.plot(cores, cores, 'k--', label='Linear Scaling (Ideal)')
plt.plot(cores, amdahl_speedup, 'b-o', label="Amdahl's Law (95% Parallel)")
plt.plot(cores, usl_speedup, 'r-s', label='Universal Scalability Law (With Crosstalk)')
plt.xlabel('Number of CPU Cores', fontsize=12)
plt.ylabel('Speedup Factor', fontsize=12)
plt.title('Theoretical Scaling Limits: Theory vs Reality', fontsize=14, fontweight='bold')
plt.xticks(cores)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

---

## 2. Setup & Environment Verification (10 Mins)
Verify connection to the Slurm scheduler and inspect the current cluster state on the REANNZ infrastructure.

In [ ]:
# Check Slurm cluster status
!sinfo -s

## 3. The Benchmark Script (20 Mins)
We will write a heavy matrix computation script (`compute_heavy.py`). It uses Python's `multiprocessing` to distribute tasks across a fixed problem size.

In [ ]:
%%writefile compute_heavy.py
import sys
import time
import numpy as np
from multiprocessing import Pool

def stress_core(matrix_size):
    A = np.random.rand(matrix_size, matrix_size)
    B = np.random.rand(matrix_size, matrix_size)
    np.dot(A, B)
    return True

if __name__ == '__main__':
    num_cores = int(sys.argv[1]) if len(sys.argv) > 1 else 1
    matrix_size = 1800
    total_tasks = 128  # Fixed total workload
    
    start_time = time.time()
    
    # Split the workload across the user-defined core count
    with Pool(num_cores) as p:
        p.map(stress_core, [matrix_size] * total_tasks)
        
    end_time = time.time()
    print(f"RESULTS_MARKER|CORES:{num_cores}|TIME:{end_time - start_time:.2f}")

## 4. Submitting the Scaling Tests (25 Mins)
We will write a loop to automatically generate and submit Slurm scripts requesting 1, 2, 4, 8, 16, 32, and 64 cores. 

To isolate performance correctly, we request explicit, un-shared resources using `--cpus-per-task`.

In [ ]:
import os
import time

core_counts = [1, 2, 4, 8, 16, 32, 64]
job_ids = {}

for cores in core_counts:
    slurm_script = f"""#!/bin/bash
#SBATCH --job-name=scaling_{cores}
#SBATCH --cpus-per-task={cores}
#SBATCH --ntasks=1
#SBATCH --time=00:10:00
#SBATCH --output=job_{cores}.out

module purge
module load python/3.11

python3 compute_heavy.py {cores}
"""
    filename = f"submit_{cores}.slurm"
    with open(filename, "w") as f:
        f.write(slurm_script)
    
    # Submit job and parse ID
    stream = os.popen(f"sbatch {filename}")
    output = stream.read()
    job_id = output.split()[-1]
    job_ids[cores] = job_id
    print(f"Submitted {cores}-core job. Slurm Job ID: {job_id}")

### Queue Monitoring
Run this block repeatedly until the queue returns empty, meaning all jobs completed successfully.

In [ ]:
!squeue -u $USER